# Lecture 8: Polynomial Transition, Contours, and Optimization Controls

Covers:
- linear to polynomial fit transition (Figure 8.2 intent),
- contour/loss-landscape behavior (Section 8.4.4 style),
- optimization controls for learning rate $\alpha$ and stopping tolerance.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

rng = np.random.default_rng(10)


def make_data(n=40, noise=0.25):
    x = np.linspace(-2.3, 2.3, n)
    y_true = 1.2 + 0.8*x - 0.6*x**2
    y = y_true + rng.normal(0, noise, n)
    return x, y, y_true


def poly_design(x, degree):
    return np.vstack([x**d for d in range(degree+1)]).T


def run_gd(x, y, alpha=0.05, max_iter=120, tol=1e-5):
    X = np.vstack([np.ones_like(x), x]).T
    theta = np.zeros(2)
    path = [theta.copy()]
    n = len(x)
    for _ in range(max_iter):
        grad = (2/n) * X.T @ (X @ theta - y)
        new = theta - alpha * grad
        path.append(new.copy())
        if np.linalg.norm(new - theta) < tol:
            theta = new
            break
        theta = new
    return np.array(path)


def draw(degree=2, noise=0.25, alpha=0.05, max_iter=120, tol=1e-5):
    x, y, _ = make_data(noise=noise)

    XP = poly_design(x, degree)
    theta_poly = np.linalg.pinv(XP.T @ XP) @ XP.T @ y
    xg = np.linspace(x.min(), x.max(), 250)
    yg = poly_design(xg, degree) @ theta_poly

    path = run_gd(x, y, alpha=alpha, max_iter=max_iter, tol=tol)

    # contour in theta0-theta1 for linear model
    t0 = np.linspace(-1.5, 2.5, 100)
    t1 = np.linspace(-2.5, 2.5, 100)
    T0, T1 = np.meshgrid(t0, t1)
    J = np.zeros_like(T0)
    for i in range(T0.shape[0]):
        for j in range(T0.shape[1]):
            yhat = T0[i,j] + T1[i,j] * x
            J[i,j] = np.mean((yhat - y)**2)

    fig, axs = plt.subplots(1, 3, figsize=(16, 4.8))

    axs[0].scatter(x, y, s=20, alpha=0.85)
    axs[0].plot(xg, yg, 'r', lw=2)
    axs[0].set_title(f'Linear to polynomial (degree={degree})')

    cs = axs[1].contour(T0, T1, J, levels=20)
    axs[1].clabel(cs, inline=True, fontsize=7)
    axs[1].plot(path[:,0], path[:,1], 'r.-', ms=3)
    axs[1].set_title('Contour + gradient path')
    axs[1].set_xlabel(r'$\theta_0$')
    axs[1].set_ylabel(r'$\theta_1$')

    iters = np.arange(len(path))
    mse_path = []
    for th in path:
        yhat = th[0] + th[1]*x
        mse_path.append(np.mean((yhat-y)**2))
    axs[2].plot(iters, mse_path, 'k-', lw=2)
    axs[2].set_title('Optimization progress')
    axs[2].set_xlabel('Iteration')
    axs[2].set_ylabel('MSE')

    plt.tight_layout()
    plt.show()

ui = {
    'degree': widgets.IntSlider(description='degree', min=1, max=10, value=2, continuous_update=False),
    'noise': widgets.FloatSlider(description='noise', min=0.05, max=1.0, step=0.05, value=0.25, continuous_update=False),
    'alpha': widgets.FloatLogSlider(description='alpha', base=10, min=-3, max=-0.5, step=0.05, value=0.05, continuous_update=False),
    'max_iter': widgets.IntSlider(description='max_iter', min=10, max=400, step=10, value=120, continuous_update=False),
    'tol': widgets.FloatLogSlider(description='tol', base=10, min=-8, max=-2, step=0.2, value=1e-5, continuous_update=False),
}
widgets.interact(draw, **ui)
